In [1]:
import requests
import h5py
import numpy as np
import matplotlib.pyplot as plt
import os
import time
import glob
import pickle
from colossus.cosmology import cosmology

In [2]:
api_key = "c2cbf44234668a2497d0d7c259b13084"
headers = {"api-key": api_key}

# Base URL for TNG50-1
baseUrl = 'http://www.tng-project.org/api/TNG50-1/snapshots/99/'
data_dir = './illustris_data/'

def get(path, params=None):
    max_retry = 10

    # make HTTP GET request to path
    for n in range(max_retry):
        try:
            r = requests.get(path, params=params, headers=headers)
            # raise exception if response code is not HTTP SUCCESS (200)
            r.raise_for_status()
            break
        except requests.HTTPError:
            if n < max_retry-1:
                time.sleep(min(2**n, 30))
                continue
            else:
                raise TimeoutError()

    if r.headers['content-type'] == 'application/json':
        return r.json() # parse json responses automatically

    if 'content-disposition' in r.headers:
        filename = data_dir + r.headers['content-disposition'].split("filename=")[1]
        with open(filename, 'wb') as f:
            f.write(r.content)
        f.close()
        return filename # return the filename string

    return r

In [3]:
cosmo = cosmology.setCosmology('illustris')
h = cosmo.h

**Extract Halo Information**

In [ ]:
def subhalo_catalog(primary_halo):
    """
    Constructs a list of the subhalos of primary_halo. Each list element is a dictionary with the following keys
    id: the subhalo's Subfind ID
    pos: a list of the subhalo's position within the simulation box (box coordinates, in units of kpc)
    mass: the subhalo's mass (in units of M_sun)
    """
    
    snapshot = get(primary_halo['meta']['snapshot'])
    z = snapshot['redshift']
    a = 1/(1+z)

    catalog = []

    all_subhalos = get(primary_halo['meta']['url'],{'limit':primary_halo['child_subhalos']['count']})
    subhalo_list = all_subhalos['child_subhalos']['results']

    for i in subhalo_list:
        subhalo = get(subhalo_list[i]['url'])
        sh_mass = subhalo['mass'] * h * 10e10 # illustris mass is in 10^10 M_sun/h
        sh_pos = np.array([h*a*subhalo['pos_x'], h*a*subhalo['pos_y'], h*a*subhalo['pos_z']]) # illustris coordinates are comoving, units kpc/h
        sh_info = {"id": subhalo['id'], "pos": sh_pos, "mass":sh_mass}
        catalog.append(sh_info)

    return catalog

**Select Milky Way-Like Halos**

In [ ]:
# find a MW-like primary halo:
halo_index=66
while True:
    primary_halo = get(baseUrl+f'halos/{halo_index}/')
    halo_info = get(primary_halo['meta']['info'])

    halo_mass = h*halo_info['GroupMassType'][1] # get mass of dark matter halo component (account for hubble parameter factor)

    if 100 <= halo_mass <= 150: # Illustris gives mass in units of 10^10 M_sun
        print(f"MW-like halo found at index {halo_index}!")

    if halo_index > 150:
        break
    else:
        halo_index += 1
        continue

# NOTE: range of MW-like halos is halo indices 66 to ~150. no need to run this cell again.

In [ ]:
def stream_halos(primary_halo, subhalo_catalog):
    """
    Get a catalog of halos that are positioned to perturb stellar streams
    """

    snapshot = get(primary_halo['meta']['snapshot'])
    z = snapshot['redshift']
    a = 1/(1+z)

    primary_data = get(primary_halo['meta']['info'])
    halo_center = h*a*np.array(primary_data['GroupPos'])

    stream_subhalos = []

    for halo in subhalo_catalog:
        position = halo['pos']
        mass = halo['mass']
        r = np.sqrt(np.sum((position-halo_center)**2))

        halo['R'] = r

        # NOTE: these cuts were copied from @shooting4thestars code; I have no idea how accurate they are
        if (1e6 < mass < 1e9) & (25 < r < 100):
            stream_subhalos.append(halo)
    
    return stream_subhalos    

In [ ]:
halo_indices = range(66,141)

n_stream_perturbers = []

for i in halo_indices:
    primary_halo = get(baseUrl+f'halos/{i}/')
    sh_cat = subhalo_catalog(primary_halo)
    stream_cat = stream_halos(primary_halo, sh_cat)

    # Volume of spherical shell where halos can perturb stellar streams
    # make sure to change this if changing the radius cuts in the above cell
    V_stream_shell = 4/3 * np.pi * (100**3 - 25 **3) # units should be kpc^3, I think

    n_stream_halos = len(stream_cat)/V_stream_shell

    n_stream_perturbers.append(n_stream_halos)